In [1]:
import pandas as pd

books = pd.read_csv("../data/processed/books_master.csv")

print(books.shape)

for col in [
    "description",
    "categories",
    "Book-Author",
    "Publisher"
]:
    print(
        col,
        books[col].notna().sum()
    )

C:\Users\anshu\AppData\Local\Temp\ipykernel_40632\3655324878.py:3: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv("../data/processed/books_master.csv")


(271360, 13)
description 692
categories 700
Book-Author 271358
Publisher 271358


In [2]:
import pandas as pd

ratings = pd.read_csv("../data/processed/ratings_master.csv")

print(ratings.shape)
ratings.head()

(1031136, 3)


,User-ID,ISBN,Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [3]:
ratings_explicit = ratings[ratings["Rating"] > 0].copy()

print(ratings_explicit.shape)

user_counts = ratings_explicit["User-ID"].value_counts()

active_users = user_counts[user_counts >= 10].index

ratings_cf = ratings_explicit[
    ratings_explicit["User-ID"].isin(active_users)
].copy()

print(ratings_cf.shape)
print(ratings_cf["User-ID"].nunique())

book_counts = ratings_cf["ISBN"].value_counts()

active_books = book_counts[book_counts >= 5].index

ratings_dense = ratings_cf[
    ratings_cf["ISBN"].isin(active_books)
].copy()

print(ratings_dense.shape)
print("Users :", ratings_dense["User-ID"].nunique())
print("Books :", ratings_dense["ISBN"].nunique())

from sklearn.preprocessing import LabelEncoder
user_encoder = LabelEncoder()
book_encoder = LabelEncoder()

ratings_dense["user_idx"] = user_encoder.fit_transform(
    ratings_dense["User-ID"]
)

ratings_dense["book_idx"] = book_encoder.fit_transform(
    ratings_dense["ISBN"]
)


print(ratings_dense["user_idx"].min())
print(ratings_dense["user_idx"].max())

print(ratings_dense["book_idx"].min())
print(ratings_dense["book_idx"].max())

n_users = ratings_dense["user_idx"].max() + 1
n_books = ratings_dense["book_idx"].max() + 1

print(n_users)
print(n_books)



(383842, 3)
(261899, 3)
6589
(108344, 3)
Users : 6436
Books : 9009
0
6435
0
9008
6436
9009


In [4]:
books = pd.read_csv(
    "../data/processed/books_master.csv",
    low_memory=False
)

books_cf = books[
    books["ISBN"].isin(
        ratings_dense["ISBN"].unique()
    )
].copy()

print("CF Books:", len(books_cf))

print(
    "Descriptions:",
    books_cf["description"].notna().sum()
)

print(
    "Categories:",
    books_cf["categories"].notna().sum()
)

CF Books: 9009
Descriptions: 193
Categories: 197


In [5]:
content_books = books[
    books["description"].notna() |
    books["categories"].notna()
].copy()

print(content_books.shape)

(716, 13)


In [6]:
content_books["content"] = (
    content_books["Book-Author"].fillna("") + " " +
    content_books["Book-Author"].fillna("") + " " +
    content_books["categories"].fillna("") + " " +
    content_books["description"].fillna("")
)

In [7]:
content_books[
    ["Book-Title", "Book-Author", "content"]
].head(3)

,Book-Title,Book-Author,content
18,The Testament,John Grisham,John Grisham John Grisham Fiction #1 NEW YORK ...
52,The Street Lawyer,JOHN GRISHAM,JOHN GRISHAM JOHN GRISHAM Fiction Michael Broc...
205,Four Blind Mice,James Patterson,James Patterson James Patterson Fiction In thi...


In [8]:
print(
    (content_books["content"].str.len() > 0).sum()
)

716


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(
    content_books["content"]
)

print(tfidf_matrix.shape)

(716, 5000)


In [10]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(
    tfidf_matrix
)

print(similarity_matrix.shape)

(716, 716)


In [11]:
print(similarity_matrix.min())
print(similarity_matrix.max())

0.0
1.000000000000005


In [12]:
book_to_idx = pd.Series(
    content_books.index,
    index=content_books["Book-Title"]
).to_dict()

In [13]:
def get_similar_books(
    title,
    content_books,
    similarity_matrix,
    top_n=5
):

    matches = content_books[
        content_books["Book-Title"]
        .str.lower()
        .str.contains(title.lower(), na=False)
    ]

    if len(matches) == 0:
        return None

    idx = matches.index[0]

    row_pos = content_books.index.get_loc(idx)

    sims = similarity_matrix[row_pos]

    similar_idx = sims.argsort()[::-1][1:top_n+1]

    return content_books.iloc[
        similar_idx
    ][
        ["Book-Title", "Book-Author"]
    ]

In [14]:
content_books[
    content_books["Book-Title"]
    .str.contains(
        "Harry",
        case=False,
        na=False
    )
][["Book-Title"]].head(20)

,Book-Title
72188,Harry Potter and the Goblet of Fire
82526,Harry Potter and the Prisoner of Azkaban
200073,Harry Potter and the Philosopher's Stone
205143,Harry Potter and the Philosopher's Stone
234605,Harry Potter and the Sorcerer's Stone
259611,Harry Potter and the Sorcerer's Stone


In [15]:
content_books[
    content_books["Book-Title"]
    .str.contains(
        "Da Vinci",
        case=False,
        na=False
    )
][["Book-Title"]]

,Book-Title
748,The Da Vinci Code
2586,The Da Vinci Code
71604,The Da Vinci Code
165320,The Da Vinci Code
252066,The Da Vinci Code
258666,The Da Vinci Code


In [16]:
content_books[
    content_books["Book-Title"]
    .str.contains(
        "Testament",
        case=False,
        na=False
    )
][["Book-Title"]]

,Book-Title
18,The Testament
949,The Testament
4315,The Testament
130507,The Testament
178775,The Testament


In [17]:
content_books[
    content_books["Book-Title"]
    .str.contains(
        "Goblet of Fire",
        case=False,
        na=False
    )
][["Book-Title","Book-Author"]]

,Book-Title,Book-Author
72188,Harry Potter and the Goblet of Fire,J. K. Rowling


In [18]:
def get_similar_books_exact(
    title,
    content_books,
    similarity_matrix,
    top_n=10
):

    idx = content_books[
        content_books["Book-Title"] == title
    ].index[0]

    row_pos = content_books.index.get_loc(idx)

    sims = similarity_matrix[row_pos]

    similar_idx = sims.argsort()[::-1][1:top_n+1]

    return content_books.iloc[
        similar_idx
    ][
        ["Book-Title", "Book-Author"]
    ]

In [19]:
get_similar_books_exact(
    "Harry Potter and the Goblet of Fire",
    content_books,
    similarity_matrix,
    top_n=10
)

,Book-Title,Book-Author
82526,Harry Potter and the Prisoner of Azkaban,J. K. Rowling
205143,Harry Potter and the Philosopher's Stone,J.K. Rowling
200073,Harry Potter and the Philosopher's Stone,J.K. Rowling
259611,Harry Potter and the Sorcerer's Stone,J.K. Rowling
234605,Harry Potter and the Sorcerer's Stone,J. K. Rowling
3914,Quidditch Through the Ages,J. K. Rowling
6385,The Eyes of the Dragon,Stephen King
56463,The Eyes of the Dragon,Stephen King
107546,Sourcery,Terry Pratchett
22013,The Long Walk,Stephen King


In [20]:
get_similar_books_exact(
    "The Testament",
    content_books,
    similarity_matrix,
    top_n=10
)

,Book-Title,Book-Author
4315,The Testament,John Grisham
178775,The Testament,John Grisham
130507,The Testament,John Grisham
949,The Testament,John Grisham
220771,The Rainmaker,JOHN GRISHAM
48228,The Rainmaker,John Grisham
184731,The Rainmaker,John Grisham
2552,The Rainmaker,John Grisham
76067,The Rainmaker,John Grisham
1012,The Rainmaker,JOHN GRISHAM


In [21]:
get_similar_books_exact(
    "The Da Vinci Code",
    content_books,
    similarity_matrix,
    top_n=10
)

,Book-Title,Book-Author
748,The Da Vinci Code,Dan Brown
2586,The Da Vinci Code,DAN BROWN
252066,The Da Vinci Code,Dan Brown
165320,The Da Vinci Code,Dan Brown
71604,The Da Vinci Code,DAN BROWN
89900,Angels and Demons,Dan Brown
11174,Illuminati.,Dan Brown
59738,America's Garden Book,Louise Bush-Brown
81338,Digital Fortress,Dan Brown
193429,Invaders from Earth,Robert Silverberg


In [22]:
content_books_unique = (
    content_books
    .drop_duplicates(
        subset=["Book-Title", "Book-Author"]
    )
    .reset_index(drop=True)
)

In [23]:
print(len(content_books))
print(len(content_books_unique))

716
373


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(
    content_books_unique["content"]
)

print(tfidf_matrix.shape)

(373, 5000)


In [25]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(
    tfidf_matrix
)

print(similarity_matrix.shape)

(373, 373)


In [26]:
get_similar_books_exact(
    "The Da Vinci Code",
    content_books_unique,
    similarity_matrix,
    top_n=10
)

,Book-Title,Book-Author
17,The Da Vinci Code,Dan Brown
259,Angels and Demons,Dan Brown
114,Illuminati.,Dan Brown
250,Digital Fortress,Dan Brown
210,America's Garden Book,Louise Bush-Brown
332,Invaders from Earth,Robert Silverberg
4,Deception Point,Dan Brown
207,Rich Dad's Guide to Becoming Rich...Without Cu...,Robert T. Kiyosaki
8,Life Before Man,Margaret Atwood
167,"Rich Dad's Retire Young, Retire Rich",Robert T. Kiyosaki


In [27]:
get_similar_books_exact(
    "Harry Potter and the Goblet of Fire",
    content_books_unique,
    similarity_matrix,
    top_n=10
)

,Book-Title,Book-Author
252,Harry Potter and the Prisoner of Azkaban,J. K. Rowling
335,Harry Potter and the Philosopher's Stone,J.K. Rowling
353,Harry Potter and the Sorcerer's Stone,J. K. Rowling
364,Harry Potter and the Sorcerer's Stone,J.K. Rowling
64,Quidditch Through the Ages,J. K. Rowling
80,The Eyes of the Dragon,Stephen King
279,Sourcery,Terry Pratchett
148,The Long Walk,Stephen King
138,Eric,Terry Pratchett
151,Dancing Girls and Other Stories,Margaret Atwood


In [28]:
semantic_isbns = set(content_books_unique["ISBN"])

active_isbns = set(
    ratings_dense["ISBN"].unique()
)

print(
    len(
        semantic_isbns &
        active_isbns
    )
)

122


In [29]:
title_to_pos = {
    title: idx
    for idx, title in enumerate(
        content_books_unique["Book-Title"]
    )
}

In [30]:
def get_neighbors(
    row_pos,
    similarity_matrix,
    top_n=10
):

    sims = similarity_matrix[row_pos]

    neighbors = (
        sims.argsort()[::-1][1:top_n+1]
    )

    return neighbors

In [31]:
semantic_graph = {}

for row_pos in range(
    len(content_books_unique)
):

    isbn = content_books_unique.iloc[
        row_pos
    ]["ISBN"]

    semantic_graph[isbn] = get_neighbors(
        row_pos,
        similarity_matrix,
        top_n=10
    )

In [32]:
content_books_unique["author_clean"] = (
    content_books_unique["Book-Author"]
    .fillna("")
    .str.lower()
    .str.replace(r"[^a-z0-9]", "", regex=True)
)

In [33]:
content_books_unique[
    content_books_unique["Book-Title"]
    .str.contains(
        "Harry Potter",
        case=False,
        na=False
    )
][["Book-Title","Book-Author","author_clean"]]

,Book-Title,Book-Author,author_clean
232,Harry Potter and the Goblet of Fire,J. K. Rowling,jkrowling
252,Harry Potter and the Prisoner of Azkaban,J. K. Rowling,jkrowling
335,Harry Potter and the Philosopher's Stone,J.K. Rowling,jkrowling
353,Harry Potter and the Sorcerer's Stone,J. K. Rowling,jkrowling
364,Harry Potter and the Sorcerer's Stone,J.K. Rowling,jkrowling


In [34]:
content_books_unique["content"] = (
    content_books_unique["author_clean"] + " " +
    content_books_unique["categories"].fillna("") + " " +
    content_books_unique["description"].fillna("")
)

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(
    content_books_unique["content"]
)

similarity_matrix = cosine_similarity(
    tfidf_matrix
)

In [36]:
get_similar_books_exact(
    "Harry Potter and the Goblet of Fire",
    content_books_unique,
    similarity_matrix,
    top_n=10
)

,Book-Title,Book-Author
252,Harry Potter and the Prisoner of Azkaban,J. K. Rowling
335,Harry Potter and the Philosopher's Stone,J.K. Rowling
364,Harry Potter and the Sorcerer's Stone,J.K. Rowling
353,Harry Potter and the Sorcerer's Stone,J. K. Rowling
80,The Eyes of the Dragon,Stephen King
279,Sourcery,Terry Pratchett
148,The Long Walk,Stephen King
138,Eric,Terry Pratchett
164,Dancing Girls: And Other Stories,Margaret Atwood
151,Dancing Girls and Other Stories,Margaret Atwood


In [37]:
isbn_to_row = {
    isbn: idx
    for idx, isbn in enumerate(
        content_books_unique["ISBN"]
    )
}

In [38]:
semantic_neighbors = {}

for row in range(len(content_books_unique)):

    sims = similarity_matrix[row]

    top_idx = sims.argsort()[::-1][1:11]

    semantic_neighbors[
        content_books_unique.iloc[row]["ISBN"]
    ] = [
        (
            content_books_unique.iloc[i]["ISBN"],
            sims[i]
        )
        for i in top_idx
    ]

In [39]:
isbn_to_row = {
    isbn: idx
    for idx, isbn in enumerate(
        content_books_unique["ISBN"]
    )
}

semantic_neighbors = {}

for row in range(len(content_books_unique)):

    sims = similarity_matrix[row]

    top_idx = sims.argsort()[::-1][1:11]

    semantic_neighbors[
        content_books_unique.iloc[row]["ISBN"]
    ] = [
        (
            content_books_unique.iloc[i]["ISBN"],
            float(sims[i])
        )
        for i in top_idx
    ]

print(len(semantic_neighbors))

373


In [40]:
print("ratings_dense" in globals())
print("train_df" in globals())
print("test_df" in globals())

True
False
False


In [41]:
print(ratings_dense.shape)

(108344, 5)


In [42]:
ratings_dense = ratings_dense.sort_values(
    ["user_idx"]
)

In [43]:
train_parts = []
test_parts = []

for _, group in ratings_dense.groupby("user_idx"):

    group = group.copy()

    test_parts.append(
        group.iloc[-1:]
    )

    train_parts.append(
        group.iloc[:-1]
    )

train_df = pd.concat(
    train_parts,
    ignore_index=True
)

test_df = pd.concat(
    test_parts,
    ignore_index=True
)

In [44]:
user_history = (
    train_df
    .groupby("User-ID")["ISBN"]
    .apply(set)
    .to_dict()
)

print(len(user_history))

6197


In [45]:
users_with_semantic_books = 0

for user, books_seen in user_history.items():

    if any(
        isbn in semantic_neighbors
        for isbn in books_seen
    ):
        users_with_semantic_books += 1

print("Users with semantic books:",
      users_with_semantic_books)

print("Total users:",
      len(user_history))

print(
    "Coverage:",
    users_with_semantic_books /
    len(user_history)
)

Users with semantic books: 1711
Total users: 6197
Coverage: 0.27610133935775377


In [46]:
from collections import defaultdict

def recommend_content(
    user_id,
    user_history,
    semantic_neighbors,
    top_k=10
):

    scores = defaultdict(float)

    seen_books = user_history.get(
        user_id,
        set()
    )

    for isbn in seen_books:

        if isbn not in semantic_neighbors:
            continue

        for neighbor_isbn, sim_score in semantic_neighbors[isbn]:

            if neighbor_isbn in seen_books:
                continue

            scores[neighbor_isbn] += sim_score

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked[:top_k]

In [47]:
sample_user = None

for user, books_seen in user_history.items():

    if any(
        isbn in semantic_neighbors
        for isbn in books_seen
    ):
        sample_user = user
        break

print(sample_user)

243


In [48]:
recs = recommend_content(
    sample_user,
    user_history,
    semantic_neighbors,
    top_k=10
)

print(recs)

[('038550120X', 1.0000000000000002), ('0316693006', 0.17853821678706663), ('0446611212', 0.15785067007731068), ('0316969443', 0.12965337189864817), ('0440200563', 0.10176041506282121), ('0671868691', 0.08825306125718255), ('1570424535', 0.0878992435448708), ('0452274664', 0.08686562385064647), ('0061099678', 0.08632484535410427), ('0679408142', 0.08546066647299107)]


In [49]:
isbn_to_title = (
    books[["ISBN", "Book-Title"]]
    .drop_duplicates()
    .set_index("ISBN")["Book-Title"]
    .to_dict()
)

In [50]:
for isbn, score in recs:
    print(
        isbn_to_title.get(isbn, "Unknown"),
        round(score, 3)
    )

A Painted House 1.0
Four Blind Mice 0.179
Violets Are Blue 0.158
Suzanne's Diary for Nicholas 0.13
Fine Things 0.102
Bitter Harvest 0.088
Miracle on the 17th Green 0.088
The Autobiography of My Mother 0.087
\My Husband's Trying to Kill Me!\"" 0.086
The World Is My Home 0.085


In [51]:
def evaluate_hit10_content(
    test_df,
    user_history,
    semantic_neighbors
):

    hits = 0
    evaluated_users = 0

    for _, row in test_df.iterrows():

        user = row["User-ID"]
        true_book = row["ISBN"]

        recs = recommend_content(
            user,
            user_history,
            semantic_neighbors,
            top_k=10
        )

        if len(recs) == 0:
            continue

        evaluated_users += 1

        rec_books = [
            isbn
            for isbn, _
            in recs
        ]

        if true_book in rec_books:
            hits += 1

    hit10 = (
        hits / evaluated_users
        if evaluated_users > 0
        else 0
    )

    return hits, evaluated_users, hit10

In [52]:
hits, users_eval, hit10 = (
    evaluate_hit10_content(
        test_df,
        user_history,
        semantic_neighbors
    )
)

print("Hits:", hits)
print("Evaluated Users:", users_eval)
print("Hit@10:", hit10)

Hits: 16
Evaluated Users: 1711
Hit@10: 0.009351256575102279


In [53]:
print("item_similarity_df" in globals())
print("model" in globals())

False
False


In [54]:
print("ratings_dense" in globals())
print("train_df" in globals())
print("test_df" in globals())
print("semantic_neighbors" in globals())
print("item_similarity_df" in globals())
print("model" in globals())

True
True
True
True
False
False


In [55]:
import pickle

with open(
    "../models/semantic_neighbors.pkl",
    "wb"
) as f:
    pickle.dump(
        semantic_neighbors,
        f
    )

print("Saved semantic neighbors")

Saved semantic neighbors


In [56]:
with open(
    "../models/isbn_to_book_idx.pkl",
    "wb"
) as f:
    pickle.dump(
        isbn_to_book_idx,
        f
    )

NameError: name 'isbn_to_book_idx' is not defined

In [ ]:
with open(
    "../models/book_idx_to_isbn.pkl",
    "wb"
) as f:
    pickle.dump(
        book_idx_to_isbn,
        f
    )

NameError: name 'book_idx_to_isbn' is not defined

In [ ]:
print(type(semantic_neighbors))
print(len(semantic_neighbors))

<class 'dict'>
373


In [ ]:
import pickle

with open(
    "../models/semantic_neighbors.pkl",
    "wb"
) as f:
    pickle.dump(
        semantic_neighbors,
        f
    )

print("Saved!")

Saved!


In [ ]:
print("semantic_neighbors" in globals())

True
